In [0]:
%pip install openpyxl

In [0]:
dbutils.library.restartPython()

In [0]:
# Recebo o caminho do arquivo RAW como parâmetro.
# Mantenho o arquivo atual como padrão para facilitar testes manuais no notebook.
dbutils.widgets.text(
    "caminho_arquivo",
    "/Volumes/workspace/raw/anp_arquivos/anp/f40da41fecc036126f442d6aa0e1bb423b7261aff8d8ea47fbe701b3c1ac778b/revendas_lpc_2026-09-06_2026-09-12.xlsx",
    "Caminho do arquivo RAW"
)

caminho_arquivo = dbutils.widgets.get("caminho_arquivo").strip()

print("Arquivo recebido:")
print(caminho_arquivo)

In [0]:
import os

# Antes de processar os dados, confirmo que o arquivo está acessível no Volume.
arquivo_encontrado = os.path.exists(caminho_arquivo)

print("Arquivo encontrado:", arquivo_encontrado)

In [0]:
import hashlib


def calcular_hash_sha256(caminho):
    """
    Calculo o SHA-256 do arquivo em blocos para não precisar
    carregar todo o conteúdo na memória de uma única vez.
    """
    hash_arquivo = hashlib.sha256()

    with open(caminho, "rb") as arquivo:
        for bloco in iter(lambda: arquivo.read(1024 * 1024), b""):
            hash_arquivo.update(bloco)

    return hash_arquivo.hexdigest()


hash_calculado = calcular_hash_sha256(caminho_arquivo)

print("SHA-256 calculado:")
print(hash_calculado)

In [0]:
import pandas as pd

# Leio apenas algumas linhas para validar a planilha e o cabeçalho
# antes de fazer a carga completa da camada RAW.
amostra = pd.read_excel(
    caminho_arquivo,
    sheet_name="POSTOS REVENDEDORES",
    header=9,
    nrows=5,
    engine="openpyxl"
)

display(amostra)

In [0]:
# Crio o schema que vai representar a camada Bronze do projeto.
spark.sql("""
    CREATE SCHEMA IF NOT EXISTS workspace.bronze
    COMMENT 'Camada Bronze com os dados estruturados da ANP antes das regras de qualidade e transformação.'
""")

print("Schema Bronze criado ou já existente.")

In [0]:
# Confirmo que o schema está disponível no Unity Catalog.
display(
    spark.sql("SHOW SCHEMAS IN workspace")
)

In [0]:
import pandas as pd

# Mantenho como texto os campos que funcionam como identificadores ou descrições.
# Isso evita que CNPJ, CEP e outros valores sejam tratados como números.
colunas_texto = [
    "CNPJ",
    "RAZÃO",
    "FANTASIA",
    "ENDEREÇO",
    "NÚMERO",
    "COMPLEMENTO",
    "BAIRRO",
    "CEP",
    "MUNICÍPIO",
    "ESTADO",
    "BANDEIRA",
    "PRODUTO",
    "UNIDADE DE MEDIDA"
]

tipos_colunas = {
    coluna: "string"
    for coluna in colunas_texto
}

# Agora faço a leitura completa do arquivo original armazenado na RAW.
dados_bronze = pd.read_excel(
    caminho_arquivo,
    sheet_name="POSTOS REVENDEDORES",
    header=9,
    dtype=tipos_colunas,
    engine="openpyxl"
)

# Garanto explicitamente os tipos das duas colunas de negócio restantes.
dados_bronze["PREÇO DE REVENDA"] = pd.to_numeric(
    dados_bronze["PREÇO DE REVENDA"],
    errors="coerce"
)

dados_bronze["DATA DA COLETA"] = pd.to_datetime(
    dados_bronze["DATA DA COLETA"],
    errors="coerce"
)

print("Registros carregados:", len(dados_bronze))
print("Colunas:", len(dados_bronze.columns))

In [0]:
# Registro a linha original do Excel para conseguir rastrear cada registro até o arquivo recebido da ANP.
dados_bronze["linha_origem"] = (
    dados_bronze.index + 11
).astype("int64")

print("Primeira linha:", dados_bronze["linha_origem"].min())
print("Última linha:", dados_bronze["linha_origem"].max())
print("Colunas:", len(dados_bronze.columns))

In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType,
    TimestampType,
    LongType
)


# Defino explicitamente o schema para não depender da inferência automática dos tipos durante a conversão para Spark.
esquema_bronze = StructType([
    StructField("CNPJ", StringType(), True),
    StructField("RAZÃO", StringType(), True),
    StructField("FANTASIA", StringType(), True),
    StructField("ENDEREÇO", StringType(), True),
    StructField("NÚMERO", StringType(), True),
    StructField("COMPLEMENTO", StringType(), True),
    StructField("BAIRRO", StringType(), True),
    StructField("CEP", StringType(), True),
    StructField("MUNICÍPIO", StringType(), True),
    StructField("ESTADO", StringType(), True),
    StructField("BANDEIRA", StringType(), True),
    StructField("PRODUTO", StringType(), True),
    StructField("UNIDADE DE MEDIDA", StringType(), True),
    StructField("PREÇO DE REVENDA", DoubleType(), True),
    StructField("DATA DA COLETA", TimestampType(), True),
    StructField("linha_origem", LongType(), False)
])

# Troco os valores ausentes do Pandas por None antes de enviar os dados ao Spark.
dados_para_spark = (
    dados_bronze
    .astype(object)
    .where(pd.notna(dados_bronze), None)
)

df_bronze = spark.createDataFrame(
    dados_para_spark,
    schema=esquema_bronze
)

print("Registros no Spark:", df_bronze.count())
print("Colunas:", len(df_bronze.columns))

In [0]:
from uuid import uuid4
from pyspark.sql.functions import (
    current_timestamp,
    lit,
    col
)
import os

from pyspark.sql.functions import (
    current_timestamp,
    lit
)

# Cada processamento no Databricks recebe seu próprio identificador.
id_execucao = str(uuid4())

nome_tabela_bronze = "workspace.bronze.revendas_anp"

# Se o arquivo já existe na Bronze, reutilizo o ID que ele já possui.
# Para um arquivo novo, gero o próximo ID disponível.
if spark.catalog.tableExists(nome_tabela_bronze):
    arquivo_existente = (
        spark.table(nome_tabela_bronze)
        .filter(col("hash_sha256") == hash_calculado)
        .select("id_arquivo")
        .limit(1)
        .first()
    )

    if arquivo_existente:
        id_arquivo = arquivo_existente["id_arquivo"]
    else:
        maior_id = (
            spark.table(nome_tabela_bronze)
            .selectExpr("MAX(id_arquivo) AS maior_id")
            .first()["maior_id"]
        )

        id_arquivo = (maior_id or 0) + 1

else:
    id_arquivo = 1

# Extraio o nome diretamente do caminho recebido pelo pipeline.
# Assim não preciso alterar o código quando chegar um novo arquivo da ANP.
arquivo_origem = os.path.basename(caminho_arquivo)

# Acrescento os metadados necessários para rastrear a origem desta carga.
df_bronze = (
    df_bronze
    .withColumn("id_execucao", lit(id_execucao))
    .withColumn("id_arquivo", lit(id_arquivo))
    .withColumn("arquivo_origem", lit(arquivo_origem))
    .withColumn("data_ingestao_utc", current_timestamp())
)

print("ID da execução:", id_execucao)
print("Registros:", df_bronze.count())
print("Colunas:", len(df_bronze.columns))
print("ID do arquivo:", id_arquivo)

In [0]:
df_bronze.printSchema()

display(
    df_bronze.limit(10)
)

In [0]:
# Uso o hash calculado diretamente do arquivo RAW.
# Assim o pipeline não depende de um valor preenchido manualmente.
hash_sha256 = hash_calculado

df_bronze = df_bronze.withColumn(
    "hash_sha256",
    lit(hash_sha256)
)

print("Hash associado à carga:")
print(hash_sha256)

print("Registros:", df_bronze.count())
print("Colunas:", len(df_bronze.columns))

In [0]:
from pyspark.sql.functions import col

nome_tabela_bronze = "workspace.bronze.revendas_anp"

# Primeiro verifico se a tabela Bronze já existe.
tabela_bronze_existe = spark.catalog.tableExists(
    nome_tabela_bronze
)

if not tabela_bronze_existe:
    # Na primeira carga, crio a tabela Delta mantendo
    # os nomes originais recebidos da ANP.
    df_bronze.createOrReplaceTempView("vw_carga_bronze_anp")

    spark.sql(f"""
        CREATE TABLE {nome_tabela_bronze}
        USING DELTA
        TBLPROPERTIES (
            'delta.columnMapping.mode' = 'name'
        )
        AS
        SELECT *
        FROM vw_carga_bronze_anp
    """)

    print("Tabela Bronze criada.")
    print("Registros adicionados:", df_bronze.count())

else:
    # Se a tabela já existe, verifico pelo hash se este
    # mesmo arquivo já foi processado.
    arquivo_ja_processado = (
        spark.table(nome_tabela_bronze)
        .filter(col("hash_sha256") == hash_calculado)
        .limit(1)
        .count()
        > 0
    )

    if arquivo_ja_processado:
        print("Arquivo já processado.")
        print("Carga Bronze ignorada para evitar duplicidade.")

    else:
        (
            df_bronze
            .write
            .mode("append")
            .saveAsTable(nome_tabela_bronze)
        )

        print("Nova carga adicionada à Bronze.")
        print("Registros adicionados:", df_bronze.count())

In [0]:
display(
    spark.sql("""
        SELECT
            COUNT(*) AS registros,
            COUNT(DISTINCT hash_sha256) AS arquivos_distintos,
            MIN(linha_origem) AS primeira_linha,
            MAX(linha_origem) AS ultima_linha
        FROM workspace.bronze.revendas_anp
    """)
)

In [0]:
display(
    spark.sql("""
        SELECT
            id_arquivo,
            arquivo_origem,
            COUNT(*) AS registros
        FROM workspace.bronze.revendas_anp
        GROUP BY
            id_arquivo,
            arquivo_origem
        ORDER BY id_arquivo
    """)
)